# Modelos interpretables para el valor por AGEB

Réplica del Boston Housing para la zona metropolitana de Guadalajara con modelos que se pueden leer, siguiendo los temas de Análisis Estadístico Multivariado, más tres interpretables de fuera del temario, una red neuronal chica y dos cajas negras de referencia. Al final, si un promedio municipal oficial basta para dar el nivel de un municipio sin precios por AGEB.

- **Objetivo:** `log(medv)`, la mediana del precio de oferta por vivienda en la AGEB (casas y departamentos en venta), en pesos de julio de 2024. Sale de los anuncios que junta el IIEG en seis municipios, repartidos a las AGEB por el área de su colonia (dataset v8, `docs/producto/dataset-vivienda.md`). El municipio entra como factor.
- **Validación:** cada AGEB se predice con un modelo que no vio su bloque de 3 km. Los hiperparámetros también se eligen por bloques.
- **Dos medidas:** RMSE en log (error relativo típico) y Spearman dentro de cada municipio (si el orden de las AGEB sale bien).

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from precio import calibracion, conformal, datos, graficas, interpretables, modelos

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)

RAIZ = Path("../..")
DATASET = RAIZ / "mapo_core/data/vivienda_gdl_v8.zip"
ALFA = 0.2  # rango con 80 % de cobertura nominal
RASGOS = datos.RASGOS

## $\color{#dda}{\text{1. Datos y correlaciones}}$

Las AGEB con al menos tres anuncios efectivos y sus 44 rasgos. Antes de modelar: qué rasgos se mueven con el precio.

In [2]:
v = datos.cargar(DATASET)
v["bloque"] = datos.bloques(v)
base = v.dropna(subset=["y"]).reset_index(drop=True)
X, y, g = base[RASGOS + ["municipio"]], base["y"], base["bloque"]
base.groupby("municipio").agg(agebs=("cvegeo", "size"), mediana=("medv", "median"),
                              anuncios=("medv_n", "sum"), pct_departamentos=("pct_departamentos", "median"))

,agebs,mediana,anuncios,pct_departamentos
municipio,,,,
El Salto,5,8.500e+05,23.11,10.00
Guadalajara,299,2.703e+06,7759.25,42.00
San Pedro Tlaquepaque,68,2.150e+06,741.16,1.75
Tlajomulco de Zúñiga,84,1.362e+06,2167.68,1.90
Tonalá,35,1.408e+06,198.98,3.40
Zapopan,296,3.500e+06,11034.95,19.70


In [3]:
graficas.correlaciones(base, RASGOS)

In [4]:
z = interpretables.estandarizar(base, RASGOS)
fuertes = base[RASGOS].corrwith(base["y"], method="spearman").abs().nlargest(12).index
graficas.mapa_calor(z[fuertes].corr(), "Correlación entre los 12 rasgos más ligados al valor", "r")

## $\color{#dda}{\text{2. Distribución del objetivo}}$

El precio en pesos tiene cola larga a la derecha; en logaritmo queda cerca de simétrico. Por eso se modela `log(precio)` y los coeficientes se leen como porcentajes.

In [5]:
pd.DataFrame({"sesgo": [base["medv"].skew(), base["y"].skew()],
              "curtosis": [base["medv"].kurt(), base["y"].kurt()]},
             index=["precio ($)", "log(precio)"])

,sesgo,curtosis
precio ($),4.919,35.754
log(precio),-0.006,1.366


In [6]:
graficas.distribucion(base, "Precio de oferta por vivienda por AGEB (escala logarítmica)", "Pesos de julio de 2024")

## $\color{#dda}{\text{3. Regresión lineal por máxima verosimilitud}}$

OLS de log(precio) con los 44 rasgos estandarizados y el municipio como factor. Si el error es normal, OLS es el estimador de máxima verosimilitud. Los errores estándar van agrupados por bloque de 3 km: dos AGEB vecinas no son dos observaciones independientes, y tratarlas así daría intervalos demasiado angostos.

In [7]:
res = interpretables.ols(base, RASGOS)
print(f"R² = {res.rsquared:.3f}, log-verosimilitud = {res.llf:.1f}, AIC = {res.aic:.1f}, BIC = {res.bic:.1f}")
coef = interpretables.coeficientes(res, RASGOS)
coef.head(15)

R² = 0.777, log-verosimilitud = -298.6, AIC = 697.1, BIC = 930.5


,coef,bajo,alto,p,cambio_pct
dis_km,-0.633,-0.783,-0.482,1.905e-16,-46.884
negocios_1km,-0.326,-0.476,-0.176,2.101e-05,-27.803
nox,-0.291,-0.438,-0.145,9.415e-05,-25.281
salud_1km,0.180,0.068,0.291,1.564e-03,19.690
pro_ocup_c,-0.166,-0.265,-0.067,1.073e-03,-15.293
pct_viv_auto,0.162,0.056,0.267,2.597e-03,17.557
pct_sin_secundaria,0.138,0.044,0.232,3.908e-03,14.839
pct_viv_3ymas_cuartos,-0.130,-0.192,-0.068,3.673e-05,-12.226
diversidad_giros_1km,0.126,0.076,0.176,6.901e-07,13.424
escuelas_1km,-0.111,-0.215,-0.008,3.534e-02,-10.527


In [8]:
graficas.coeficientes(coef.head(20), "OLS completa: efecto de cada rasgo (los 20 más grandes)")

In [9]:
graficas.qq(res.resid, "Residuos de la OLS contra la normal")

Con 44 rasgos correlacionados muchos signos son inestables. El VIF mide cuánto se infla la varianza de cada coeficiente por la colinealidad (arriba de 10 es grave).

In [10]:
interpretables.vif(base, RASGOS).head(12)

age                       53.031
age_1975                  51.600
viv_ha                    24.589
viv_hab                   23.860
negocios_1km              21.073
dis_km                    18.133
salud_1km                 16.088
pct_viv_internet          13.970
area_ha                   13.366
nox                       13.323
graproes                  12.919
restaurantes_cafes_1km    12.285
Name: vif, dtype: float64

## $\color{#dda}{\text{4. Selección de variables}}$

Dos caminos. **Por pasos:** se agrega el rasgo que más baja el BIC hasta que ninguno lo baje. **Lasso:** penaliza el tamaño de los coeficientes y deja en cero los que no aportan; la penalización se elige por validación cruzada.

In [11]:
pasos = interpretables.Pasos(RASGOS).fit(X, y)
elegidos = pasos.elegidos_
print(len(elegidos), "rasgos por pasos:", elegidos)
res_pasos = interpretables.ols(base, elegidos)
coef_pasos = interpretables.coeficientes(res_pasos, elegidos)
coef_pasos

15 rasgos por pasos: ['pro_ocup_c', 'lst_c', 'viv_ha', 'dis_km', 'pct_viv_auto', 'pct_sin_secundaria', 'pct_viv_3ymas_cuartos', 'nox', 'pobtot', 'altitud_m', 'diversidad_giros_1km', 'negocios_1km', 'salud_1km', 'pct_sin_servicios_salud', 'rad_km']


,coef,bajo,alto,p,cambio_pct
dis_km,-0.614,-0.762,-0.466,4.785e-16,-45.868
negocios_1km,-0.387,-0.522,-0.252,2.046e-08,-32.094
nox,-0.309,-0.436,-0.182,1.795e-06,-26.566
salud_1km,0.239,0.139,0.339,2.762e-06,27.002
pct_viv_auto,0.212,0.122,0.302,4.215e-06,23.624
pro_ocup_c,-0.202,-0.288,-0.115,5.600e-06,-18.250
viv_ha,-0.167,-0.224,-0.110,1.194e-08,-15.385
lst_c,-0.138,-0.177,-0.100,1.195e-12,-12.922
diversidad_giros_1km,0.134,0.088,0.181,1.696e-08,14.363
pct_sin_secundaria,0.133,0.063,0.204,2.146e-04,14.235


In [12]:
graficas.coeficientes(coef_pasos, "OLS con los rasgos elegidos por BIC")

In [13]:
interpretables.vif(base, elegidos)

negocios_1km               14.410
salud_1km                  10.681
dis_km                      9.469
nox                         7.554
pro_ocup_c                  6.196
pct_viv_auto                4.825
pct_sin_secundaria          4.158
pct_sin_servicios_salud     2.589
pobtot                      2.320
diversidad_giros_1km        2.283
viv_ha                      2.089
pct_viv_3ymas_cuartos       2.065
lst_c                       1.791
altitud_m                   1.585
rad_km                      1.400
Name: vif, dtype: float64

In [14]:
cat = interpretables.catalogo(RASGOS)
lasso = modelos.ajustar(cat["Lasso"][1], X, y, g)
coef_lasso = interpretables.coef_lasso(lasso)
print(f"Lasso deja {coef_lasso.index.isin(RASGOS).sum()} de {len(RASGOS)} rasgos, más {(~coef_lasso.index.isin(RASGOS)).sum()} indicadores de dato faltante")
coef_lasso.head(20)

Lasso deja 38 de 44 rasgos, más 18 indicadores de dato faltante


,coef,cambio_pct
dis_km,-0.556,-42.625
negocios_1km,-0.283,-24.623
nox,-0.253,-22.391
viv_hab,-0.234,-20.827
pobtot,0.227,25.456
pct_sin_secundaria,0.218,24.384
pro_ocup_c,-0.198,-17.981
pct_viv_auto,0.185,20.375
salud_1km,0.153,16.578
graproes,0.121,12.845


## $\color{#dda}{\text{5. ANOVA y ANCOVA por municipio}}$

¿Cuánto del valor explica solo el municipio? ANOVA de una vía. Después, ANCOVA: el municipio con los rasgos elegidos como covariables. Al final, si la pendiente del rasgo más fuerte cambia entre municipios (el supuesto de pendientes paralelas de la ANCOVA).

In [15]:
anova = interpretables.anova_municipio(base, elegidos)
anova["una_via"]

,sum_sq,df,F,PR(>F),eta2
C(municipio),79.184,5.0,34.125,1.389e-31,0.179
Residual,362.442,781.0,NaN,NaN,NaN


In [16]:
anova["ancova"]

,sum_sq,df,F,PR(>F)
C(municipio),7.275,5.0,10.577,7.720e-10
pro_ocup_c,4.720,1.0,34.313,6.963e-09
lst_c,8.305,1.0,60.374,2.523e-14
viv_ha,10.114,1.0,73.521,5.474e-17
dis_km,16.602,1.0,120.689,3.568e-26
pct_viv_auto,7.107,1.0,51.664,1.566e-12
pct_sin_secundaria,3.145,1.0,22.864,2.086e-06
pct_viv_3ymas_cuartos,3.524,1.0,25.619,5.213e-07
nox,6.539,1.0,47.536,1.130e-11
pobtot,3.073,1.0,22.340,2.717e-06


In [17]:
anova["pendientes"]

,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,766.0,105.372,0.0,NaN,NaN,NaN
1,761.0,102.676,5.0,2.696,3.996,0.001


## $\color{#dda}{\text{6. Componentes principales}}$

Los 44 rasgos comparten mucha información. PCA los resume en ejes sin correlación; la regresión sobre componentes (PCR) usa los primeros. Las cargas dicen qué significa cada eje.

In [18]:
var, cargas = interpretables.pca(base, RASGOS, n=3)
graficas.varianza(var)

In [19]:
top = cargas.abs().max(axis=1).nlargest(20).index
graficas.mapa_calor(cargas.loc[top], "Cargas de los tres primeros componentes", "carga")

## $\color{#dda}{\text{7. Tipos de zona con k-medias}}$

Agrupar AGEB parecidas en tipos de zona y ver si el tipo, junto con el municipio, basta para explicar el valor (una ANOVA de dos factores).

In [20]:
etiqueta, perfil = interpretables.tipos(base, RASGOS, k=6)
base.assign(tipo=etiqueta).groupby("tipo").agg(agebs=("cvegeo", "size"), mediana=("medv", "median"))

,agebs,mediana
tipo,,
0,255,2.500e+06
1,85,9.990e+05
2,146,3.800e+06
3,7,4.164e+06
4,195,4.490e+06
5,99,2.350e+06


In [21]:
destaca = perfil.abs().max(axis=1).nlargest(18).index
graficas.mapa_calor(perfil.loc[destaca], "Perfil de cada tipo (media estandarizada)", "z")

## $\color{#dda}{\text{8. Regresión logística: qué hace cara a una AGEB}}$

Otra pregunta, más fácil de contar: la probabilidad de que una AGEB valga más que la mediana de su municipio. Razón de momios por una desviación estándar de cada rasgo elegido; arriba de 1, sube la probabilidad.

In [22]:
momios, auc = interpretables.logistica(base, elegidos)
print(f"AUC fuera de muestra por bloques: {auc:.3f}")
momios

AUC fuera de muestra por bloques: 0.838


,razon_momios,bajo,alto,p
negocios_1km,0.186,0.091,0.380,4.049e-06
viv_ha,0.382,0.274,0.534,1.599e-08
dis_km,0.454,0.184,1.120,8.659e-02
pro_ocup_c,0.513,0.302,0.872,1.359e-02
pct_viv_3ymas_cuartos,0.678,0.492,0.935,1.771e-02
rad_km,0.717,0.523,0.982,3.791e-02
nox,0.862,0.420,1.767,6.847e-01
lst_c,0.875,0.683,1.122,2.935e-01
pct_sin_secundaria,0.956,0.591,1.545,8.539e-01
pobtot,1.164,0.777,1.744,4.618e-01


## $\color{#dda}{\text{9. Comparación fuera de muestra}}$

Todos con la misma validación por bloques. Familias: del temario de la clase, interpretables de fuera del temario, la red y las dos cajas negras de referencia (SVR y bosque, las del estudio principal).

In [23]:
predicciones, tablas = {}, []
for nombre, (familia, modelo) in cat.items():
    predicciones[nombre] = modelos.fuera_de_muestra(modelo, X, y, g)
    tablas.append(modelos.metricas(y, predicciones[nombre], base["municipio"]).assign(modelo=nombre))
comparacion = pd.concat(tablas)
familias = {n: f for n, (f, _) in cat.items()}
comparacion.pivot(index="modelo", columns="municipio", values="rmse_log").sort_values("Todos")

municipio,El Salto,Guadalajara,San Pedro Tlaquepaque,Tlajomulco de Zúñiga,Todos,Tonalá,Zapopan
modelo,,,,,,,
Lasso,0.292,0.373,0.391,0.442,0.398,0.394,0.413
Mixto,0.303,0.370,0.396,0.422,0.403,0.395,0.432
Bosque,0.348,0.347,0.365,0.468,0.403,0.449,0.438
OLS (ANCOVA),0.340,0.370,0.398,0.424,0.404,0.388,0.434
Aditivo (splines),0.275,0.355,0.392,0.484,0.406,0.428,0.431
Pasos BIC,0.445,0.372,0.417,0.477,0.407,0.406,0.416
SVR,0.404,0.350,0.367,0.536,0.422,0.496,0.454
Red (32-16),0.417,0.379,0.447,0.574,0.460,0.549,0.489
PCR,0.394,0.454,0.444,0.579,0.470,0.458,0.458


In [24]:
comparacion.pivot(index="modelo", columns="municipio", values="spearman").sort_values("Todos", ascending=False)

municipio,El Salto,Guadalajara,San Pedro Tlaquepaque,Tlajomulco de Zúñiga,Todos,Tonalá,Zapopan
modelo,,,,,,,
Lasso,0.975,0.675,0.562,0.917,0.828,0.450,0.822
Aditivo (splines),0.975,0.675,0.506,0.907,0.827,0.357,0.814
OLS (ANCOVA),0.975,0.687,0.533,0.924,0.825,0.470,0.803
Mixto,0.975,0.686,0.534,0.924,0.824,0.459,0.803
Pasos BIC,0.975,0.690,0.481,0.906,0.823,0.391,0.820
Bosque,0.975,0.681,0.447,0.917,0.820,0.177,0.806
SVR,0.821,0.689,0.558,0.866,0.814,0.432,0.795
Red (32-16),0.975,0.660,0.440,0.853,0.764,0.476,0.739
PCR,0.975,0.532,0.344,0.867,0.754,0.203,0.781


In [25]:
graficas.modelos_juntos(comparacion.query("municipio == 'Todos'"), familias)

In [26]:
arbol = modelos.ajustar(cat["Árbol"][1], X, y, g)
print(interpretables.reglas_arbol(arbol, RASGOS))

|--- pro_ocup_c <= 0.70
|   |--- lst_c <= 44.95
|   |   |--- ndvi <= 0.30
|   |   |   |--- altitud_m <= 1608.00
|   |   |   |   |--- value: [15.71]
|   |   |   |--- altitud_m >  1608.00
|   |   |   |   |--- value: [16.07]
|   |   |--- ndvi >  0.30
|   |   |   |--- value: [16.57]
|   |--- lst_c >  44.95
|   |   |--- graproes <= 13.57
|   |   |   |--- diversidad_giros_1km <= 2.55
|   |   |   |   |--- value: [14.73]
|   |   |   |--- diversidad_giros_1km >  2.55
|   |   |   |   |--- negocios_1km <= 7.36
|   |   |   |   |   |--- value: [15.22]
|   |   |   |   |--- negocios_1km >  7.36
|   |   |   |   |   |--- value: [14.94]
|   |   |--- graproes >  13.57
|   |   |   |--- ndvi <= 0.16
|   |   |   |   |--- indus <= 0.13
|   |   |   |   |   |--- value: [15.37]
|   |   |   |   |--- indus >  0.13
|   |   |   |   |   |--- value: [15.06]
|   |   |   |--- ndvi >  0.16
|   |   |   |   |--- pct_viv_2ymas_dorm <= 84.03
|   |   |   |   |   |--- transporte_km <= 1.13
|   |   |   |   |   |   |--- value: 

**Transferencia:** se deja fuera un municipio entero y se mide el orden dentro de él. Es lo que haría falta para dar forma a un municipio sin tabla de valores.

In [27]:
candidatos = ["Lasso", "Aditivo (splines)", "PCR", "Mixto", "Red (32-16)", "Bosque"]
pd.concat([modelos.transferencia(cat[n][1], base, RASGOS).assign(modelo=n) for n in candidatos]
          ).pivot(index="municipio", columns="modelo", values="spearman")

modelo,Aditivo (splines),Bosque,Lasso,Mixto,PCR,Red (32-16)
municipio,,,,,,
Guadalajara,0.497,0.654,0.582,0.594,0.529,0.660
San Pedro Tlaquepaque,0.462,0.515,0.568,0.543,0.423,0.266
Tlajomulco de Zúñiga,0.871,0.892,0.917,0.921,0.862,0.811
Tonalá,0.393,0.278,0.498,0.494,0.294,0.467
Zapopan,0.739,0.734,0.779,0.744,0.706,0.656


## $\color{#dda}{\text{10. Rango conformal con un modelo interpretable}}$

El rango conformal no depende del modelo. Se compara el del estudio principal (bosque para el centro y para el ancho) contra Lasso para el centro con el ancho de un bosque, y Lasso para el centro con el ancho de una ridge sobre log|residuo|, que también se puede leer.

In [28]:
variantes = {
    "Bosque + ancho bosque": lambda: conformal.Normalizado(cat["Bosque"][1], RASGOS),
    "Lasso + ancho bosque": lambda: conformal.Normalizado(cat["Lasso"][1], RASGOS),
    "Lasso + ancho lineal": lambda: conformal.Normalizado(cat["Lasso"][1], RASGOS, escala="lineal"),
}
rangos = {n: conformal.evaluar(f, base, RASGOS, ALFA) for n, f in variantes.items()}
cob = pd.concat([conformal.resumen(r).assign(metodo=n) for n, r in rangos.items()])
cob.pivot(index="municipio", columns="metodo", values=["cobertura", "alto_entre_bajo_mediana"])

cobertura                       \
metodo                Bosque + ancho bosque Lasso + ancho bosque   
municipio                                                          
El Salto                              1.000                1.000   
Guadalajara                           0.839                0.829   
San Pedro Tlaquepaque                 0.853                0.897   
Tlajomulco de Zúñiga                  0.750                0.774   
Todos                                 0.835                0.827   
Tonalá                                0.886                0.800   
Zapopan                               0.841                0.824   

                                           alto_entre_bajo_mediana  \
metodo                Lasso + ancho lineal   Bosque + ancho bosque   
municipio                                                            
El Salto                             1.000                   2.693   
Guadalajara                          0.826                   2.301   
San Pedro Tlaquepaque                0.809                   2.668   
Tlajomulco de Zúñiga                 0.774                   4.805   
Todos                                0.820                   2.709   
Tonalá                               0.800                   3.500   
Zapopan                              0.828                   2.886   

                                                                 
metodo                Lasso + ancho bosque Lasso + ancho lineal  
municipio                                                        
El Salto                             3.273                3.193  
Guadalajara                          2.580                2.529  
San Pedro Tlaquepaque                2.873                2.731  
Tlajomulco de Zúñiga                 3.390                3.426  
Todos                                2.862                2.781  
Tonalá                               3.497                3.373  
Zapopan                              2.925                2.893

In [29]:
graficas.cobertura(cob, 1 - ALFA)

## $\color{#dda}{\text{11. Calibrar con el promedio municipal}}$

¿Sirve una cifra municipal para dar el nivel de un municipio sin precios por AGEB? Se deja fuera un municipio entero, se predicen sus AGEB con Lasso ajustada en los demás, y se compara la mediana predicha del municipio con la mediana de **todos** los anuncios del IIEG ahí (también los que no traen colonia). Después se mueve el nivel para que coincidan y se mide el error por AGEB otra vez. La última columna es la referencia: el error de dar a todas las AGEB la mediana observada del municipio, que es lo mejor que da la cifra sola.

La mediana predicha se pesa por viviendas habitadas, lo único que habría en un municipio sin anuncios.

In [30]:
municipal = datos.municipal(DATASET)
oficial = municipal.merge(v[["cve_mun", "municipio"]].drop_duplicates(), on="cve_mun").set_index("municipio")["mediana"]
v["peso_viv"] = np.expm1(v["viv_hab"])  # datos.cargar deja viv_hab en log1p
municipal

,cve_mun,municipio_iieg,periodo,anuncios,anuncios_con_colonia,media,p25,mediana,p75
0,039,GUADALAJARA,2024-07,11884,8074,6210174,2950000,4627000,7434920
1,120,ZAPOPAN,2024-07,21509,12071,9651692,3600000,6149899,10750000
2,098,TLAQUEPAQUE,2024-07,1860,963,2730383,1580000,2450000,3300000
3,101,TONALA,2024-07,886,412,1693989,850000,1388500,2100000
4,097,TLAJOMULCO,2024-07,5608,2821,5817523,2067000,3700000,6260105
5,070,EL SALTO,2024-07,231,71,1720287,500000,850000,1850000


In [31]:
fuera = calibracion.municipio_fuera(cat["Lasso"][1], base, v, RASGOS, oficial, ponderar="peso_viv")
fuera

,municipio,agebs,oficial,predicha_sin_calibrar,error_nivel_pct,rmse_log_sin_calibrar,rmse_log_calibrado,rmse_log_con_su_mediana,agebs_sin_precio
0,Guadalajara,299,4627000,2239686,-51.595,0.423,0.860,0.529,143
1,El Salto,5,850000,1409253,65.794,0.298,0.698,1.213,122
2,Tlajomulco de Zúñiga,84,3700000,889733,-75.953,0.490,1.633,1.123,293
3,San Pedro Tlaquepaque,68,2450000,1684176,-31.258,0.445,0.435,0.399,166
4,Tonalá,35,1388500,1827899,31.646,0.509,0.376,0.423,193
5,Zapopan,296,6149899,3157635,-48.656,0.453,0.766,0.723,233


Lo mismo, pesando cada AGEB por sus anuncios efectivos: imita a la cifra oficial, que es una mediana de anuncios. En un municipio sin datos no se tendría ese peso; es la cota de lo mejor posible.

In [32]:
fuera_anuncios = calibracion.municipio_fuera(cat["Lasso"][1], base, v, RASGOS, oficial, ponderar="medv_n")
fuera_anuncios

,municipio,agebs,oficial,predicha_sin_calibrar,error_nivel_pct,rmse_log_sin_calibrar,rmse_log_calibrado,rmse_log_con_su_mediana,agebs_sin_precio
0,Guadalajara,299,4627000,6297829,36.110,0.423,0.510,0.529,143
1,El Salto,5,850000,1265230,48.851,0.298,0.600,1.213,122
2,Tlajomulco de Zúñiga,84,3700000,3643852,-1.518,0.490,0.495,1.123,293
3,San Pedro Tlaquepaque,68,2450000,1947397,-20.514,0.445,0.399,0.399,166
4,Tonalá,35,1388500,1838849,32.434,0.509,0.375,0.423,193
5,Zapopan,296,6149899,5896715,-4.117,0.453,0.450,0.723,233


In [33]:
pd.DataFrame({
    "sin calibrar": [fuera["rmse_log_sin_calibrar"].mean()],
    "calibrado (viviendas)": [fuera["rmse_log_calibrado"].mean()],
    "calibrado (anuncios)": [fuera_anuncios["rmse_log_calibrado"].mean()],
    "solo la mediana observada": [fuera["rmse_log_con_su_mediana"].mean()],
}, index=["RMSE log promedio de los municipios"])

,sin calibrar,calibrado (viviendas),calibrado (anuncios),solo la mediana observada
RMSE log promedio de los municipios,0.436,0.795,0.471,0.735


**La SHF no sirve para calibrar estos precios.** Para Jalisco, en el primer trimestre de 2026, da una media de \$2.21 millones y una mediana de \$1.28 millones por vivienda (boletín de la SHF, vía IIEG). Los anuncios de la zona metropolitana de julio de 2024 tienen una mediana de \$4.95 millones. No es que una esté mal: la SHF mide avalúos de viviendas compradas con crédito hipotecario en todo el estado, y el IIEG mide precios pedidos en portales de la zona metropolitana, donde Zapopan pone más de la mitad de los anuncios. Forzar la media de las AGEB a la de la SHF bajaría todo el mapa a un nivel que no es el de ninguna de las dos poblaciones.

## $\color{#dda}{\text{12. Conclusiones}}$

**Lasso es el mejor modelo, y es interpretable.** Fuera de muestra por bloques: RMSE en log de 0.398 y Spearman de 0.828, por delante del bosque (0.403 y 0.820), la SVR (0.422 y 0.814) y la red (0.460 y 0.764). El modelo mixto, la OLS, el aditivo y la selección por pasos quedan pegados a Lasso (de 0.403 a 0.407). Con el precio de oferta, el orden sale mucho mejor que con el valor catastral (Spearman 0.83 contra 0.74): el catastro traía el ruido del OCR y años distintos.

**Lo que mueve el precio** coincide en la OLS por pasos y la Lasso:

| Rasgo | Efecto por una desviación estándar |
|---|---|
| Distancia a los centros de empleo (`dis_km`) | −43 a −47 % |
| Negocios a 1 km | −25 a −32 % (a igual diversidad de giros) |
| NO₂ (`nox`) | −22 a −27 % |
| Unidades de salud a 1 km | +17 a +27 % |
| Viviendas con automóvil | +18 a +24 % |
| Ocupantes por cuarto | −15 a −18 % |
| Temperatura de superficie | −9 a −13 % |

Algunos signos salen al revés de lo esperado (más población sin secundaria, +14 %) porque se miden con todo lo demás fijo y las variables están muy correlacionadas: hay que leerlos junto con el VIF, no solos.

**Por tema:**

- **Distribución:** el precio tiene sesgo de 4.9; en logaritmo, 0.0. El logaritmo aquí no es opcional.
- **Regresión lineal:** la OLS completa explica 78 % de la varianza del log del precio (con el catastro era 46 %).
- **ANOVA y ANCOVA:** solo el municipio explica 18 % (η² = 0.18). A diferencia del catastro, con las covariables sigue siendo significativo (p < 0.001): a igual zona, cada municipio tiene su propio nivel de precios pedidos. Las pendientes paralelas se rechazan (p = 0.001): el efecto del hacinamiento cambia entre municipios.
- **k-medias:** seis tipos de zona con medianas de \$1.0 a \$4.5 millones. Como modelo es el peor (Spearman 0.63): un tipo es demasiado grueso.
- **Logística:** AUC fuera de muestra de 0.84 para "¿vale más que la mediana de su municipio?". Lo que más sube los momios son las unidades de salud cerca (3.5 veces por desviación estándar) y lo que más los baja, los negocios a 1 km (0.19) y la densidad de vivienda (0.38).
- **Fuera del temario:** el aditivo empata con Lasso; el árbol se lee fácil (primero hacinamiento, luego temperatura y escolaridad) pero predice mal (0.527).
- **Red neuronal:** de las peores (0.460). Con 787 AGEB, un modelo lineal penalizado le gana.
- **Transferencia:** Lasso ordena bien un municipio que no vio: Tlajomulco 0.92, Zapopan 0.78, Guadalajara 0.58, Tlaquepaque 0.57, Tonalá 0.50. Tonalá es el más débil en todo: tiene pocos anuncios por AGEB (mediana 4.8) y ahí domina el ruido.

**Rango conformal:** ahora los tres cubren lo prometido o un poco más (de 82 a 84 % contra 80 % nominal), y con Lasso y ancho lineal el rango típico va de un piso a 2.8 veces el piso, igual que el bosque. Con el precio de oferta, Lasso puede reemplazar al bosque sin perder cobertura.

**Calibrar con el promedio municipal no mejora.** Al dejar fuera un municipio y mover su nivel para que la mediana predicha iguale la mediana oficial de anuncios:

- Pesando por viviendas, que es lo único que habría en un municipio sin datos, el error por AGEB **empeora** casi en todos (Tlajomulco de 0.49 a 1.63; Guadalajara de 0.42 a 0.86). La mediana de anuncios describe dónde se anuncia (los fraccionamientos caros de Tlajomulco, las colonias de Zapopan), no la vivienda típica del municipio.
- Pesando por anuncios, el mejor caso, queda más o menos igual: mejora Tonalá y Tlaquepaque, empeora Guadalajara y El Salto. Y sin calibrar, el modelo ya acierta el nivel de Zapopan y Tlajomulco con menos de 5 % de error.

En promedio de los seis municipios, el RMSE en log es 0.44 sin calibrar, 0.80 calibrando por viviendas, 0.47 calibrando por anuncios y 0.74 con solo la mediana observada.

La cifra municipal sirve si mide la misma población que lo que se predice. Para un municipio sin anuncios por AGEB, primero habría que modelar dónde aparecen los anuncios; usar la media municipal tal cual mete un sesgo, no lo quita. Por lo mismo, la SHF (avalúos de crédito en todo el estado) no sirve para calibrar precios pedidos.